In [10]:
!pip install -q -U langchain langchain-community langchain-google-genai faiss-cpu



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import os 
import time 
import re 
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from pathlib import Path
from dotenv import load_dotenv



In [12]:
# Load .env (development) into env vars; production should set real env vars or use a secrets manager.
load_dotenv()  # reads .env if present

# Prefer explicit env var; fall back to a secrets file only if provided
api_key = os.getenv("GEMINI_API_KEY") 

if not api_key:
    secrets_path = Path(os.getenv("SECRETS_PATH", Path("secrets") / "api"))
    if secrets_path.exists():
        with secrets_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith("GEMINI_API_KEY="):
                    api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                    break

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in environment or secrets file. Set GEMINI_API_KEY or SECRETS_PATH.")

os.environ["GEMINI_API_KEY"] = api_key

print("API key loaded:", bool(api_key))



API key loaded: True


In [13]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash", temperature= 0.0)
embedder = GoogleGenerativeAIEmbeddings(model = "models/gemini-embedding-001")

In [14]:
print("[System] Refreshing Vector Database...")
audit_data = [
    Document(page_content="MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate.", metadata={"id": 1}),
    Document(page_content="Tuesday Review confirmed the 15% spike is due to 'Legacy Scan-X' hardware and firmware v2.1.", metadata={"id": 2}),
    Document(page_content="Jaymin approved a $45,000 emergency budget to upgrade European scanners by Q1 end.", metadata={"id": 3}),
    Document(page_content="OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.", metadata={"id": 4}),
    Document(page_content="Tony recommends a distributed architecture for handling 500+ PDFs in legacy branches.", metadata={"id": 5}),
    Document(page_content="The 15% error rate is classified as 'Critical' for Banking and Compliance audits.", metadata={"id": 6}),
    Document(page_content="Marten's team is monitoring OCR logs 24/7 until the hardware upgrade is finished.", metadata={"id": 7}),
    Document(page_content="European legacy branches are the only units still using the v2.1 firmware.", metadata={"id": 8})
]


[System] Refreshing Vector Database...


In [15]:
vectorstore = FAISS.from_documents(audit_data, embedder)
base_retriever = vectorstore.as_retriever(search_kwargs = {"k":3})


In [16]:
step_back_prompt = ChatPromptTemplate.from_template(
    """You are an expert at world knowledge.
    Paraphrase the following question into a SINGLE, concise, foundational step-back question.
    Output ONLY the question and nothing else. No introductions, no bullet points, no options.

    Original Question: {question}
    Step-Back Question:"""
)

step_back_chain = step_back_prompt | llm | StrOutputParser()

response_prompt = ChatPromptTemplate.from_template(
    """You are an elite MLOps auditor. Answer using ONLY this context.
    Foundational Context: {step_back_context}
    Specific Context: {normal_context}
    Question: {question}
    Final Answer:"""

)

In [ ]:
# Clean retrieval logic

def run_custom_rag(query: str):
    # step 1 : Get step back query 
    raw_broad_q = step_back_chain.invoke({"question": query})

    #logic to clean the output 
    lines = raw_broad_q.split('\n')
    broad_q = next((line.strip() for line in lines if '?' in line), lines[0].strip())

    print(f"[Debug] Cleaned Broad Query: {broad_q}")
    # step 2 : Specific Retrieval
    n_docs = base_retriever.invoke(query)

    # API cooldown 
    time.sleep(2.5)

    # Step 4: Broad Retrieval 
    sb_docs = base_retriever.invoke(broad_q)

    return {
        "step_back_context": "\n\n".join(d.page_content for d in sb_docs),
        "normal_context":"\n\n".join(d.page_content for d in n_docs),
        "question": query
    }

# Assemble and run 
advanced_chain = RunnableLambda(run_custom_rag) | response_prompt | llm | StrOutputParser()

query = "Why did the OCR failure rate specifically spike to 15% for the legacy branch offices in the European division during the Tuesday review?"

try: 
    print("\n ---- Inititiating DEEP AUDIT ---\n")
    print(advanced_chain.invoke(query))
except Exception as e:
    print(f"\n[Fatal Error]: {e}")





